# vepyr — VEP Annotation

Annotate variants from a VCF file using a pre-built Ensembl VEP parquet cache.

In [ ]:
import logging

logging.basicConfig(level=logging.INFO)

In [ ]:
import vepyr

In [ ]:
# Configure paths
VCF_INPUT = "/path/to/input.vcf.gz"  # raw VCF (may contain multiallelic variants)
CACHE_DIR = "/path/to/parquet/115_GRCh38_vep"  # from vepyr.build_cache()
REFERENCE_FASTA = "/path/to/Homo_sapiens.GRCh38.dna.primary_assembly.fa"
WORK_DIR = "/tmp/vepyr_work"

## Normalize, compress, and index the VCF

VEP requires **biallelic** variants. Use `bcftools norm -m -both` to decompose
multiallelic sites into separate records. Then bgzip-compress and tabix-index
the result for efficient random access.

In [ ]:
import os
import subprocess

os.makedirs(WORK_DIR, exist_ok=True)

vcf_norm = os.path.join(WORK_DIR, "normalized.vcf")
vcf_gz = vcf_norm + ".gz"

# Step 1: Decompose multiallelic variants into biallelic records
print("Normalizing (bcftools norm -m -both) ...")
result = subprocess.run(
    ["bcftools", "norm", "-m", "-both", "-o", vcf_norm, VCF_INPUT],
    capture_output=True, text=True,
)
print(result.stderr.strip())
assert result.returncode == 0, f"bcftools norm failed: {result.stderr}"

# Step 2: Bgzip compress
print("Compressing (bgzip) ...")
subprocess.run(["bgzip", "-f", vcf_norm], check=True)
assert os.path.exists(vcf_gz)

# Step 3: Tabix index
print("Indexing (tabix) ...")
subprocess.run(["tabix", "-p", "vcf", vcf_gz], check=True)

n_variants = int(subprocess.check_output(
    f"zcat {vcf_gz} | grep -cv '^#'", shell=True
).strip())
print(f"Ready: {n_variants:,} biallelic variants in {vcf_gz}")

VCF = vcf_gz

In [ ]:
lf = vepyr.annotate(
    VCF,
    CACHE_DIR,
    everything=True,
    reference_fasta=REFERENCE_FASTA,
    use_fjall=False
)
lf

In [ ]:
df = lf.collect()
print(f"Variants: {df.height:,}")
print(f"Columns: {df.width}")
df.head(5)

## Inspect annotation columns

In [ ]:
# Key annotation columns
df.select([
    "chrom", "start", "ref", "alt",
    "most_severe_consequence",
    "SYMBOL", "Gene", "IMPACT",
    "HGVSc", "HGVSp",
    "SIFT", "PolyPhen",
]).head(10)

In [ ]:
# Allele frequencies
df.select([
    "chrom", "start", "ref", "alt",
    "AF", "gnomADe_AF", "gnomADg_AF", "MAX_AF",
]).head(10)

In [ ]:
# Consequence distribution
df.group_by("most_severe_consequence").len().sort("len", descending=True).head(15)

## Selective annotation

Instead of `everything=True`, you can pick individual flags:

In [ ]:
# HGVS + allele frequencies only
lf_selective = vepyr.annotate(
    VCF,
    CACHE_DIR,
    hgvs=True,
    af=True,
    af_gnomadg=True,
    reference_fasta=REFERENCE_FASTA,
)
lf_selective.collect().head(5)

## Filter and export

In [ ]:
import polars as pl

# Filter to high-impact variants
high_impact = (
    lf
    .filter(pl.col("IMPACT") == "HIGH")
    .select(["chrom", "start", "ref", "alt", "SYMBOL", "most_severe_consequence", "HGVSc", "HGVSp"])
    .collect()
)
print(f"High-impact variants: {high_impact.height}")
high_impact.head(10)

In [ ]:
# Export to parquet
# df.write_parquet("/tmp/annotated.parquet")

# Export to CSV
# df.write_csv("/tmp/annotated.csv")